In [1]:
# To import relevant libraries
import numpy as np
import pandas as pd
import random
import sys
from random import randrange
from math import sqrt
from math import pi
from math import exp
from math import isnan

### Function to stratify dataset

In [2]:
# Stratifying the data into training data set and testing data set
# The testSize is specified as ratio

def trainTestSplit(data, testSize):
    # Converting the test size (ratio) into numbers of rows 
    if isinstance(testSize, float):
        testSize = round(testSize * len(data))
        
    # Getting the index of the data from the dataset
    dataIndex = data.index.tolist()
    
    # Generate random indexes to split the dataset
    testIndex = random.sample(population = dataIndex, k = testSize)
    
    # Allocate the testing set according to the random indexes (testIndex)
    testSet = data.loc[testIndex]
    
    # Dropping the testset rows from the original data set, and make the remaining rows as the training set
    trainSet = data.drop(testIndex)
    
    # Return the train and test set data
    return trainSet, testSet

### Gaussian Naive Bayes Function

In [3]:
# Calculate the Gaussian probability distribution for continuous features
# To prevent numerical underflow, when the exponent underflows is out of Pythons' float precision, set
# exponent to the smallest possible float value by the system
# To prevent the zero frequency/count problem, the function catches occurences of 0/null standard deviation
# and returns the probability of 1/number of observations

def calculateGaussianProbability(x, mean, stdev, totalRows):
    if stdev == 0 or isnan(stdev):
        return 1/totalRows # if stdev is 0, return the probability of "Add one count"
    
    exponent = exp(-((x - mean)**2 / (2 * stdev**2 )))
    
    if exponent == 0:
        # Set exponent to smallest possible float supported by the system
        exponent = sys.float_info.min
        
    return (1 / (sqrt(2 * pi) * stdev)) * exponent

### Function to calculate probability for categorical features

In [4]:
# Calculate probability for categorical features
# To prevent the zero frequency/count problem, the function catches the occurences of 0/null standard deviation

def calculateProbability(x, X1, count1, X2, count2, classCount, totalRows):
    if x == X1:
        # If zero frequency occurs, add 1 to count and return the probability
        if count1/classCount == 0:
            return 1/totalRows
        
        return count1/classCount
    else:
        # If zero frequency occurs, add 1 to count and return the probability
        if count2/classCount == 0:
            return 1/totalRows
        
        return count2/classCount
    

### Function to calculate class probability

In [5]:
# Calculate the probabilities of predicting each class for a given row:
# For continuous features use Gaussian probability function
# For categorical feature calculate_probability function

def calculateClassProbabilities(summaries, row):
    
    # Get the length of the dataset
    # Sum up all the counts of each label class
    totalRows = sum([summaries[label][0][2] for label in summaries])
    
    # Instantiate a dictionary to store probability of each label class for a given row
    probabilities = dict()
    
    # Get the class value: classValue
    # Get the summaries for each class: classSummaries
    for classValue, classSummaries in summaries.items():
        
        # Get the probability of each label class e.g., If class label 1 has a length of 12345
        # and length of dataset is 234567 then this probability is 12345/234567
        probabilities[classValue] = summaries[classValue][0][2]/float(totalRows)
        
        #looping through each summaries
        for i in range(len(classSummaries)):
            # if categorical feature
            if len(classSummaries[i]) > 3:
                X1, count1, X2, count2, classCount = classSummaries[i]
                probabilities[classValue] = \
                probabilities[classValue] * calculateProbability(row[i], X1, count1, X2, count2, classCount, totalRows)
            # if continuous feature
            else:
                mean, stdev, _ = classSummaries[i]
                probabilities[classValue] = \
                probabilities[classValue] * calculateGaussianProbability(row[i], mean, stdev, totalRows)
                
    return probabilities

### Make prediction

In [6]:
# Predict the class for a given row
def predictClass(summaries, row):
    # Storing the probabilities by calling the method
    probabilities = calculateClassProbabilities(summaries, row)
    
    # Initializing the variables
    bestLabel, bestProb = None, -1
    
    # Looping through to find the best label with the best probability
    for classVal, probability in probabilities.items():
        if bestLabel is None or probability > bestProb:
            bestProb = probability
            bestLabel = classVal
    return bestLabel

### Function to get a summary dictionary of dataset

In [7]:
def summarizeDataset(data):
    # Creating a empty dict to store the target class
    summaries = {}
    # For loop to loop the target column that are unique
    for i in data.iloc[:,-1].unique():
        # listing the feature variable of each unique class
        feature = []
        # looping through all the feature variable except the target column
        for j in range(len(data.columns)-1):
            # Storing the size of unique values in a variable
            uniqueValueSize = len(data.iloc[:,j].unique())
            
            # If categorical feature
            if(uniqueValueSize < 5):
                aList = list()
                # Creating a subset for each class
                df = data[data.iloc[:,-1] == i]
                
                # looping through the categorical feature for unique vals
                for k in data.iloc[:,j].unique():
                    # storing the unique value and counting the val
                    aList.append(k)
                    aList.append(len(df[df.iloc[:,j] == k]))
                feature.append(tuple([aList[0], aList[1], aList[2], aList[3], len(data[data.iloc[:,-1] == i])]))
            # else continuous feature
            else:
                feature.append((data[data.iloc[:,-1] == i].mean(axis = 0)[j], \
                                data[data.iloc[:,-1] == i].std(axis = 0)[j], len(data[data.iloc[:,-1] == i])))
        # storing the summaries     
        summaries[i] = feature
    return summaries

In [8]:
# Naive Bayes Classifier Algorithm to predict the train set and the test set
def naiveBayesian(trainSet, testSet):
    # Method to get the summary
    summary = summarizeDataset(trainSet)
    
    # Creating an empty list
    predictions = list()
    
    # looping through the test set values 
    for row in testSet.values:
        output = predictClass(summary, row)
        predictions.append(output)
        
    return(predictions)

### Function to determine regression metrics

In [9]:
# Calculated as:
# check for equality of predicted value and labels in test_set
# calculates the sum of correct prediction
# divides the sum by length of test_set

    
def accuracy(predictions, dataSet):
    yTest = list(dataSet.iloc[:,-1])
    correctCount = 0
    sumError = 0.0
    rsmeError = 0.0
    for i in range(len(yTest)):
        if predictions[i] == yTest[i]:
            correctCount += 1
        sumError += abs(predictions[i] - yTest[i])
        predictionError = abs(predictions[i] - yTest[i])
        rsmeError = (predictionError**2)
        
    print(f'Number of exact matches in predictions: {correctCount}/{len(yTest)}')        
    print(f'Mean Squared Error(MSE): {np.square(np.subtract(yTest,predictions)).mean()}')
    print(f'Root Mean Squared Error (RMSE): {sqrt(rsmeError/float(len(yTest)))}')
    print(f'Mean Absolute Error(MSE): {sumError/float(len(yTest))}')
    
    return (round(correctCount/len(dataSet)*100,3))

## Preprocessing

### Abalone data set

In [10]:
#Define headers
colNames = ["Sex", "Length", "Diameter", "Height", "Whole weight", 
           "Shucked weight", "Viscera weight","Shell weight", "Rings"]

#Import data from data file and separate them by delimiter ',' 
df = pd.read_csv('abalone.data', header = None, names = colNames)

#print the created dataframe 
df.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [11]:
df

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


In [12]:
# Check for missing data and type of data
df.head().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             5 non-null      object 
 1   Length          5 non-null      float64
 2   Diameter        5 non-null      float64
 3   Height          5 non-null      float64
 4   Whole weight    5 non-null      float64
 5   Shucked weight  5 non-null      float64
 6   Viscera weight  5 non-null      float64
 7   Shell weight    5 non-null      float64
 8   Rings           5 non-null      int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 488.0+ bytes


In [13]:
## Encode the Sex 
newdf = pd.Series(df['Sex'], dtype = "category")
df['Sex'] = newdf.cat.codes
df.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,2,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,2,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,0,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,2,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,1,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [14]:
# Get the statistical info of the numeric features
df.describe()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
count,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000
mean,1.052909,0.523992,0.407881,0.139516,0.828742,0.359367,0.180594,0.238831,9.933684
std,0.822240,0.120093,0.099240,0.041827,0.490389,0.221963,0.109614,0.139203,3.224169
min,0.000000,0.075000,0.055000,0.000000,0.002000,0.001000,0.000500,0.001500,1.000000
25%,0.000000,0.450000,0.350000,0.115000,0.441500,0.186000,0.093500,0.130000,8.000000
50%,1.000000,0.545000,0.425000,0.140000,0.799500,0.336000,0.171000,0.234000,9.000000
75%,2.000000,0.615000,0.480000,0.165000,1.153000,0.502000,0.253000,0.329000,11.000000
max,2.000000,0.815000,0.650000,1.130000,2.825500,1.488000,0.760000,1.005000,29.000000


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   int8   
 1   Length          4177 non-null   float64
 2   Diameter        4177 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole weight    4177 non-null   float64
 5   Shucked weight  4177 non-null   float64
 6   Viscera weight  4177 non-null   float64
 7   Shell weight    4177 non-null   float64
 8   Rings           4177 non-null   int64  
dtypes: float64(7), int64(1), int8(1)
memory usage: 265.3 KB


In [28]:
# Split the dataset into training and tetrainSet, testSet = trainTestSplit(df, 0.3)sting
trainSet, testSet = trainTestSplit(df, 0.5)
trainSet.describe()

,id,SepalLengthCM,SepalWidthCM,PetalLengthCM,PetalWidthCM,Class
count,75.000000,75.000000,75.000000,75.000000,75.000000,75.000000
mean,77.413333,5.833333,3.081333,3.792000,1.229333,1.026667
std,44.676342,0.838166,0.453423,1.782783,0.780466,0.837844
min,1.000000,4.300000,2.000000,1.100000,0.100000,0.000000
25%,41.500000,5.100000,2.800000,1.650000,0.350000,0.000000
50%,72.000000,5.800000,3.000000,4.400000,1.400000,1.000000
75%,119.500000,6.400000,3.350000,5.100000,1.800000,2.000000
max,150.000000,7.900000,4.400000,6.900000,2.500000,2.000000


In [17]:
# An insight into the test data set to see if any 
testSet.describe()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
count,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000
mean,1.041500,0.521093,0.406169,0.139242,0.821230,0.355353,0.178841,0.236681,9.873903
std,0.821944,0.121047,0.100283,0.038868,0.490458,0.218789,0.109070,0.141210,3.299733
min,0.000000,0.110000,0.090000,0.025000,0.008000,0.002500,0.000500,0.003000,3.000000
25%,0.000000,0.450000,0.345000,0.115000,0.438000,0.188500,0.092500,0.126000,8.000000
50%,1.000000,0.540000,0.420000,0.140000,0.796000,0.330500,0.172000,0.230000,9.000000
75%,2.000000,0.610000,0.480000,0.165000,1.156000,0.497000,0.252500,0.325000,11.000000
max,2.000000,0.815000,0.650000,0.250000,2.779500,1.348500,0.760000,1.005000,29.000000


In [29]:
# Test the model on training set
trainPred = naiveBayesian(trainSet, testSet)
print('Accuracy of prediction for training set:', accuracy(trainPred, trainSet))

Number of exact matches in predictions: 18/75
Mean Squared Error(MSE): 1.44
Root Mean Squared Error (RMSE): 0.0
Mean Absolute Error(MSE): 0.9866666666666667
Accuracy of prediction for training set: 24.0


In [30]:
# Test the model on testing set
testPred = naiveBayesian(trainSet, testSet)
print('Accuracy of prediction for testing set:', accuracy(testPred, testSet))

Number of exact matches in predictions: 75/75
Mean Squared Error(MSE): 0.0
Root Mean Squared Error (RMSE): 0.0
Mean Absolute Error(MSE): 0.0
Accuracy of prediction for testing set: 100.0


### Iris data set (Another example)

In [20]:
# Define headers
colNames = ["id", "SepalLengthCM", "SepalWidthCM", "PetalLengthCM", "PetalWidthCM", "Class"]

#Import data from data file and separate them by delimiter ',' 
df = pd.read_csv('iris.data', header = None, names = colNames)

#print the created dataframe 
df.head()

,id,SepalLengthCM,SepalWidthCM,PetalLengthCM,PetalWidthCM,Class
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [21]:
# Check for missing data and type of data
df.head().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             5 non-null      int64  
 1   SepalLengthCM  5 non-null      float64
 2   SepalWidthCM   5 non-null      float64
 3   PetalLengthCM  5 non-null      float64
 4   PetalWidthCM   5 non-null      float64
 5   Class          5 non-null      object 
dtypes: float64(4), int64(1), object(1)
memory usage: 368.0+ bytes


In [22]:
## Encode the Class 
newdf = pd.Series(df['Class'], dtype = "category")
df['Class'] = newdf.cat.codes
df.head(100)

,id,SepalLengthCM,SepalWidthCM,PetalLengthCM,PetalWidthCM,Class
0,1,5.1,3.5,1.4,0.2,0
1,2,4.9,3.0,1.4,0.2,0
2,3,4.7,3.2,1.3,0.2,0
3,4,4.6,3.1,1.5,0.2,0
4,5,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...,...
95,96,5.7,3.0,4.2,1.2,1
96,97,5.7,2.9,4.2,1.3,1
97,98,6.2,2.9,4.3,1.3,1
98,99,5.1,2.5,3.0,1.1,1


In [23]:
# shuffling dataset with sample
df = df.sample(frac=1, random_state=1).reset_index(drop=True)
# df shape
print(df.shape)
#handle missing values and replace them with a '0'
df.isnull().sum()

(150, 6)


id               0
SepalLengthCM    0
SepalWidthCM     0
PetalLengthCM    0
PetalWidthCM     0
Class            0
dtype: int64

In [31]:
# Split the dataset into training and testing
trainSet, testSet = trainTestSplit(df, 0.5)
trainSet.describe()

,id,SepalLengthCM,SepalWidthCM,PetalLengthCM,PetalWidthCM,Class
count,75.000000,75.000000,75.000000,75.000000,75.000000,75.000000
mean,71.453333,5.805333,3.065333,3.682667,1.180000,0.946667
std,44.392687,0.885452,0.402188,1.824338,0.794236,0.820239
min,1.000000,4.300000,2.200000,1.000000,0.100000,0.000000
25%,30.500000,5.000000,2.800000,1.600000,0.300000,0.000000
50%,71.000000,5.800000,3.000000,4.100000,1.300000,1.000000
75%,105.500000,6.450000,3.300000,5.100000,1.800000,2.000000
max,150.000000,7.700000,4.400000,6.900000,2.500000,2.000000


In [32]:
# An insight into the test data set to see if any 
testSet.describe()

,id,SepalLengthCM,SepalWidthCM,PetalLengthCM,PetalWidthCM,Class
count,75.000000,75.000000,75.000000,75.000000,75.000000,75.000000
mean,79.546667,5.881333,3.042667,3.834667,1.217333,1.053333
std,42.386348,0.770520,0.465331,1.711289,0.735651,0.820239
min,4.000000,4.500000,2.000000,1.200000,0.100000,0.000000
25%,41.500000,5.200000,2.750000,1.550000,0.350000,0.000000
50%,80.000000,5.800000,3.000000,4.500000,1.400000,1.000000
75%,116.500000,6.350000,3.350000,5.100000,1.800000,2.000000
max,148.000000,7.900000,4.200000,6.700000,2.400000,2.000000


In [26]:
# Test the model on training set
trainPred = naiveBayesian(trainSet, trainSet)
print('Accuracy of prediction for training set:', accuracy(trainPred, trainSet))

Number of exact matches in predictions: 105/105
Mean Squared Error(MSE): 0.0
Root Mean Squared Error (RMSE): 0.0
Mean Absolute Error(MSE): 0.0
Accuracy of prediction for training set: 100.0


In [27]:
# Test the model on testing set
testPred = naiveBayesian(trainSet, testSet)
print('Accuracy of prediction for testing set:', accuracy(testPred, testSet))

Number of exact matches in predictions: 44/45
Mean Squared Error(MSE): 0.022222222222222223
Root Mean Squared Error (RMSE): 0.0
Mean Absolute Error(MSE): 0.022222222222222223
Accuracy of prediction for testing set: 97.778
